In [3]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
from difflib import get_close_matches

FILE_PATH = "uncleaned dataset cutomer segmentation.xlsx"
SHEET_NAME = "uncleaned data"
REF_DATE = pd.Timestamp("2026-11-03")

df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
rows_before = len(df)
print("Shape before cleaning:", df.shape)
print(df.dtypes)
print(df.isnull().sum())



Shape before cleaning: (4000, 15)
Customer ID                    str
Transaction / Invoice ID       str
Purchase Date                  str
Product Category               str
Product Name                   str
Quantity Purchased          object
Unit Price                  object
Discount Amount             object
Income Level                   str
Total Spending Amount       object
Purchase Frequency          object
Recency of Purchase         object
cust_name                      str
LOCATION                       str
Remarks                        str
dtype: object
Customer ID                   48
Transaction / Invoice ID      20
Purchase Date                 95
Product Category             140
Product Name                 113
Quantity Purchased           102
Unit Price                   118
Discount Amount              392
Income Level                 242
Total Spending Amount        114
Purchase Frequency            79
Recency of Purchase           80
cust_name                     7

In [4]:
df = df.dropna(how="all")
df = df.drop_duplicates()

df = df.drop(columns=["Remarks"])

df.columns = df.columns.str.strip()

df = df.rename(columns={
    "Customer ID": "customer_id",
    "Transaction / Invoice ID": "transaction_id",
    "Purchase Date": "purchase_date",
    "Product Category": "product_category",
    "Product Name": "product_name",
    "Quantity Purchased": "quantity",
    "Unit Price": "unit_price",
    "Discount Amount": "discount_amount",
    "Income Level": "income_level",
    "Total Spending Amount": "total_spending_amount",
    "Purchase Frequency": "purchase_frequency",
    "Recency of Purchase": "recency_of_purchase",
    "LOCATION": "location",
})


In [5]:
df = df.replace(r"(?i)^\s*(-|\?|n/?a|nan|null|none)\s*$", np.nan, regex=True)

df["cust_name"] = df["cust_name"].str.replace(r"[^A-Za-z ]", "", regex=True)
df["product_name"] = df["product_name"].str.replace(r"[^A-Za-z0-9 \-]", "", regex=True)

text_cols = ["customer_id", "transaction_id", "product_category", "product_name",
             "income_level", "cust_name", "location"]
for col in text_cols:

    df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)


df["customer_id"] = df["customer_id"].str.upper()
df["transaction_id"] = df["transaction_id"].str.upper().str.replace("-", "", regex=False)

df["product_category"] = df["product_category"].str.lower()
df["income_level"] = df["income_level"].str.lower()
df["product_name"] = df["product_name"].str.lower()

category_map = {
    "electronics": "Electronics", "electronic": "Electronics", "electonics": "Electronics", "electrnics": "Electronics",
    "clothing": "Clothing", "clothes": "Clothing", "cloathing": "Clothing",
    "groceries": "Groceries", "grocery": "Groceries", "grocerys": "Groceries",
    "home & kitchen": "Home & Kitchen", "home&kitchen": "Home & Kitchen", "home kitchen": "Home & Kitchen",
    "home and kitchen": "Home & Kitchen",
    "beauty": "Beauty", "beuty": "Beauty",
    "sports": "Sports", "sport": "Sports", "sprots": "Sports",
}
valid_categories = ["Electronics", "Clothing", "Groceries", "Home & Kitchen", "Beauty", "Sports"]
lower_valid = {c.lower(): c for c in valid_categories}

def fix_category(x):
    if pd.isna(x):
        return np.nan
    if x in category_map:
        return category_map[x]
    match = get_close_matches(x, list(lower_valid.keys()), n=1, cutoff=0.7)
    return lower_valid[match[0]] if match else np.nan

df["product_category"] = df["product_category"].apply(fix_category)

income_map = {
    "low": "Low", "lo": "Low", "low income": "Low",
    "medium": "Medium", "med": "Medium", "mid": "Medium",
    "high": "High", "hi": "High", "high income": "High",
}
df["income_level"] = df["income_level"].map(income_map)

product_master = {
    "Wireless Mouse": "Electronics", "Bluetooth Speaker": "Electronics", "USB Cable": "Electronics",
    "Headphones": "Electronics", "Power Bank": "Electronics",
    "Cotton T-Shirt": "Clothing", "Denim Jeans": "Clothing", "Running Shoes": "Clothing",
    "Winter Jacket": "Clothing", "Formal Shirt": "Clothing",
    "Basmati Rice 5kg": "Groceries", "Olive Oil 1L": "Groceries", "Green Tea Pack": "Groceries",
    "Almonds 500g": "Groceries", "Organic Honey": "Groceries",
    "Non-stick Pan": "Home & Kitchen", "Water Bottle": "Home & Kitchen", "Mixer Grinder": "Home & Kitchen",
    "Bed Sheet Set": "Home & Kitchen", "Dinner Set": "Home & Kitchen",
    "Face Cream": "Beauty", "Shampoo": "Beauty", "Perfume": "Beauty", "Lipstick": "Beauty", "Sunscreen": "Beauty",
    "Yoga Mat": "Sports", "Cricket Bat": "Sports", "Dumbbell Set": "Sports", "Football": "Sports",
    "Badminton Racket": "Sports",
}
lower_to_proper = {name.lower(): name for name in product_master}

def fix_product(name):
    if pd.isna(name):
        return np.nan
    if name in lower_to_proper:
        return lower_to_proper[name]
    match = get_close_matches(name, list(lower_to_proper.keys()), n=1, cutoff=0.8)
    return lower_to_proper[match[0]] if match else np.nan

df["product_name"] = df["product_name"].apply(fix_product)

df["cust_name"] = df["cust_name"].str.title()
df["location"] = df["location"].str.replace(r"\s*\|\s*", ", ", regex=True).str.title()


In [6]:
def parse_date(x):
    if pd.isna(x):
        return pd.NaT
    if isinstance(x, (pd.Timestamp, datetime)):
        return pd.Timestamp(x)
    s = str(x).strip()
    for fmt in ("%d-%m-%Y",   # 22-06-2026
                "%Y-%m-%d",   # 2025-03-01
                "%Y/%m/%d",   # 2025/03/01
                "%m/%d/%Y",   # 08/15/2025  (month first, as in this file)
                "%d.%m.%y",   # 19.11.24
                "%b %d, %Y"): # Mar 17, 2025
        try:
            return pd.Timestamp(datetime.strptime(s, fmt))
        except ValueError:
            continue
    return pd.NaT

df["purchase_date"] = df["purchase_date"].apply(parse_date)
df.loc[df["purchase_date"] > REF_DATE, "purchase_date"] = pd.NaT


In [7]:
word_to_num = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5}

def to_number(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x).lower().strip()
    if s in word_to_num:
        return float(word_to_num[s])
    s = re.sub(r"rs\.?|₹|inr|units?|,", "", s).strip()
    try:
        return float(s)
    except ValueError:
        return np.nan

num_cols = ["quantity", "unit_price", "discount_amount", "total_spending_amount",
            "purchase_frequency", "recency_of_purchase"]
for col in num_cols:
  
    df[col] = df[col].apply(to_number)


In [8]:
df.loc[df["quantity"] <= 0, "quantity"] = np.nan
df.loc[df["unit_price"] <= 0, "unit_price"] = np.nan
df.loc[df["total_spending_amount"] <= 0, "total_spending_amount"] = np.nan
df.loc[df["discount_amount"] < 0, "discount_amount"] = np.nan
df.loc[df["purchase_frequency"] < 1, "purchase_frequency"] = np.nan
df.loc[(df["recency_of_purchase"] < 0) | (df["recency_of_purchase"] > 731), "recency_of_purchase"] = np.nan

def iqr_bounds(s):
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

q_low, q_high = iqr_bounds(df["quantity"])
df.loc[(df["quantity"] < q_low) | (df["quantity"] > q_high), "quantity"] = np.nan
p_low, p_high = iqr_bounds(df["unit_price"])
df.loc[(df["unit_price"] < p_low) | (df["unit_price"] > p_high), "unit_price"] = np.nan
total_cap = q_high * p_high
df.loc[df["total_spending_amount"] > total_cap, "total_spending_amount"] = np.nan


In [9]:
def fill_money(d):
    d["unit_price"] = d["unit_price"].fillna(((d["total_spending_amount"] + d["discount_amount"]) / d["quantity"]).round(2))
    d["quantity"] = d["quantity"].fillna(((d["total_spending_amount"] + d["discount_amount"]) / d["unit_price"]).round().clip(lower=1))
    d["discount_amount"] = d["discount_amount"].fillna((d["quantity"] * d["unit_price"] - d["total_spending_amount"]).clip(lower=0).round(2))
    d["total_spending_amount"] = d["total_spending_amount"].fillna((d["quantity"] * d["unit_price"] - d["discount_amount"]).round(2))
    return d

df = fill_money(df)

typical_price = df.groupby("product_name")["unit_price"].median()
for i in df.index[df["product_name"].isna() & df["product_category"].isin(valid_categories) & df["unit_price"].notna()]:
    candidates = [p for p in typical_price.index if product_master[p] == df.at[i, "product_category"]]
    df.at[i, "product_name"] = min(candidates, key=lambda p: abs(typical_price[p] - df.at[i, "unit_price"]))

df["product_category"] = df["product_name"].map(product_master).fillna(df["product_category"])

df["unit_price"] = df["unit_price"].fillna(df["product_name"].map(df.groupby("product_name")["unit_price"].median()))
df["quantity"] = df["quantity"].fillna(1)
df = fill_money(df)
df["discount_amount"] = df["discount_amount"].fillna(0)
df = fill_money(df)

calc_total = (df["quantity"] * df["unit_price"] - df["discount_amount"]).round(2)
df["total_spending_amount"] = np.where((df["total_spending_amount"] - calc_total).abs() > 1,
                                       calc_total, df["total_spending_amount"])


In [10]:
keys = ["cust_name", "purchase_frequency", "recency_of_purchase"]
known = df.dropna(subset=keys + ["customer_id"])
lookup = (known.groupby(keys)["customer_id"]
          .agg(lambda s: s.iloc[0] if s.nunique() == 1 else np.nan)
          .rename("recovered_id").reset_index())
df = df.merge(lookup, on=keys, how="left")
df["customer_id"] = df["customer_id"].fillna(df["recovered_id"])
df = df.drop(columns="recovered_id")

for col in ["income_level", "cust_name", "location", "purchase_frequency", "recency_of_purchase"]:
    mode_value = df.groupby("customer_id")[col].transform(lambda s: s.mode().iloc[0] if s.notna().any() else np.nan)
   
    df[col] = df[col].fillna(mode_value)

df["income_level"] = df["income_level"].fillna("Unknown")
df["cust_name"] = df["cust_name"].fillna("Unknown")
df["location"] = df["location"].fillna("Unknown")

df = df.dropna(subset=["customer_id", "purchase_date", "product_name", "product_category",
                       "quantity", "unit_price", "total_spending_amount"])

df["purchase_frequency"] = df["purchase_frequency"].fillna(df.groupby("customer_id")["transaction_id"].transform("nunique"))
latest_purchase = df.groupby("customer_id")["purchase_date"].transform("max")
df["recency_of_purchase"] = df["recency_of_purchase"].fillna((REF_DATE - latest_purchase).dt.days)

df = df.drop_duplicates(subset=["transaction_id"], keep="first")

df[["first_name", "last_name"]] = df["cust_name"].str.split(" ", n=1, expand=True)
df["last_name"] = df["last_name"].fillna("Unknown")
df[["city", "state"]] = df["location"].str.split(",", n=1, expand=True)
df["state"] = df["state"].str.strip().fillna("Unknown")
df = df.drop(columns=["cust_name", "location"])


In [11]:
for col in ["quantity", "purchase_frequency", "recency_of_purchase"]:
    df[col] = df[col].round().astype(int)
for col in ["unit_price", "discount_amount", "total_spending_amount"]:
    df[col] = df[col].round(2)

df = df[["customer_id", "transaction_id", "purchase_date", "product_category", "product_name",
         "quantity", "unit_price", "discount_amount", "income_level", "total_spending_amount",
         "purchase_frequency", "recency_of_purchase", "first_name", "last_name", "city", "state"]]

df = df.sort_values(["purchase_date", "transaction_id"]).reset_index(drop=True)



In [12]:
def check(name, condition):
    print(("PASS  " if condition else "FAIL  ") + name)

check("No null values", df.isnull().sum().sum() == 0)
check("No duplicate rows", not df.duplicated().any())
check("transaction_id is unique", df["transaction_id"].is_unique)
check("customer_id format", df["customer_id"].str.fullmatch(r"C\d{4}").all())
check("transaction_id format", df["transaction_id"].str.fullmatch(r"INV\d{6}").all())
check("purchase_date valid", pd.api.types.is_datetime64_any_dtype(df["purchase_date"]) and (df["purchase_date"] <= REF_DATE).all())
check("6 valid categories", set(df["product_category"]) == set(product_master.values()))
check("product matches category", (df["product_name"].map(product_master) == df["product_category"]).all())
check("income labels valid", df["income_level"].isin(["Low", "Medium", "High", "Unknown"]).all())
check("quantity >= 1", (df["quantity"] >= 1).all())
check("unit_price > 0", (df["unit_price"] > 0).all())
check("discount >= 0", (df["discount_amount"] >= 0).all())
check("frequency >= 1", (df["purchase_frequency"] >= 1).all())
check("recency 0-731", df["recency_of_purchase"].between(0, 731).all())
check("total = qty x price - discount", ((df["quantity"] * df["unit_price"] - df["discount_amount"] - df["total_spending_amount"]).abs() <= 0.05).all())
check("one income per customer", (df[df["income_level"] != "Unknown"].groupby("customer_id")["income_level"].nunique() == 1).all())
check("one frequency per customer", (df.groupby("customer_id")["purchase_frequency"].nunique() == 1).all())


PASS  No null values
PASS  No duplicate rows
PASS  transaction_id is unique
PASS  customer_id format
PASS  transaction_id format
PASS  purchase_date valid
PASS  6 valid categories
PASS  product matches category
PASS  income labels valid
PASS  quantity >= 1
PASS  unit_price > 0
PASS  discount >= 0
PASS  frequency >= 1
PASS  recency 0-731
PASS  total = qty x price - discount
PASS  one income per customer
PASS  one frequency per customer


In [13]:

print("Rows before:", rows_before, "| Rows after:", len(df), "| Columns:", df.shape[1])
df.insert(0, "row_no", range(1, len(df) + 1))
df.columns = ["Row_no", "Customer_ID", "Transaction_ID", "Purchase_Date", "Product_Category",
              "Product_Name", "Quantity", "Unit_Price", "Discount_Amount", "Income_Level",
              "Total_Spending_Amount", "Purchase_Frequency", "Recency_of_Purchase",
              "First_Name", "Last_Name", "City", "State"]
df.to_excel("Customer_Segmentation_Cleaned_Dataset.xlsx", sheet_name="cleaned dataset", index=False)
df.to_csv("Customer_Segmentation_Cleaned_Dataset.csv", index=False)
df.head(10)


Rows before: 4000 | Rows after: 3750 | Columns: 16


,Row_no,Customer_ID,Transaction_ID,Purchase_Date,Product_Category,Product_Name,Quantity,Unit_Price,Discount_Amount,Income_Level,Total_Spending_Amount,Purchase_Frequency,Recency_of_Purchase,First_Name,Last_Name,City,State
0,1,C1251,INV100156,2024-11-04,Groceries,Green Tea Pack,2,232.00,69.60,Low,394.40,10,29,Vikram,Lee,Chennai,Tamil Nadu
1,2,C1172,INV100650,2024-11-04,Electronics,Wireless Mouse,1,926.48,0.00,Low,926.48,6,85,Ravi,Lim,Coimbatore,Tamil Nadu
2,3,C1164,INV100729,2024-11-04,Sports,Badminton Racket,2,1492.97,597.19,Low,2388.75,4,312,Sneha,Iyer,Hyderabad,Telangana
3,4,C1395,INV100799,2024-11-04,Sports,Yoga Mat,2,852.55,85.26,High,1619.84,13,80,Aarav,Nair,Bengaluru,Karnataka
4,5,C1202,INV101917,2024-11-04,Groceries,Almonds 500g,2,549.33,109.87,Low,988.79,3,257,Lakshmi,Kumar,Coimbatore,Tamil Nadu
5,6,C1159,INV102889,2024-11-04,Electronics,Headphones,1,1751.40,0.00,Medium,1751.40,11,66,John,Smith,Coimbatore,Tamil Nadu
6,7,C1591,INV102956,2024-11-04,Clothing,Formal Shirt,2,1320.50,0.00,Medium,2641.00,38,29,Nisha,Patel,Delhi,Delhi
7,8,C1339,INV103552,2024-11-04,Sports,Yoga Mat,1,948.92,189.78,Low,759.14,41,7,Sophia,Tan,Bengaluru,Karnataka
8,9,C1694,INV100695,2024-11-05,Home & Kitchen,Mixer Grinder,2,3461.39,346.14,Medium,6576.64,14,26,Anjali,Brown,Kochi,Kerala
9,10,C1419,INV102808,2024-11-05,Sports,Badminton Racket,1,1660.18,0.00,Low,1660.18,123,3,Fatima,Patel,Delhi,Delhi
